# AmsiNet - Step 22: confusion rates for Section 5.1 (CPU, about 1 minute)
Reads the saved word-level probabilities (no training). Prints, for flat ViT-B/16 and the EfficientNet-B0 word agent, the confusion rates on validation and test (5 seeds pooled) for the patterns described in the paper. Run all and paste the output back.

In [ ]:
from google.colab import drive
import os, json, numpy as np, pandas as pd
if not os.path.ismount('/content/drive'): drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/amsinet'
D = json.load(open(f'{ROOT}/data/bstd_index.json', encoding='utf-8')); SC = D['scripts']; SI = {s:i for i,s in enumerate(SC)}; C = len(SC)
META = pd.read_csv(f'{ROOT}/data/crops_meta.csv')
def conf(tag, sp):
    d = META[META.split == sp].reset_index(drop=True); y = np.array([SI[s] for s in d.script]); M = np.zeros((C, C))
    for s in range(5):
        f = f'{ROOT}/agents/{tag}_seed{s}_{sp}_probs.npy'
        if not os.path.exists(f): continue
        P = np.load(f); pa = pd.read_csv(f'{ROOT}/agents/{tag}_seed{s}_{sp}_paths.csv').path.tolist()
        assert pa == d.path.tolist(), 'misaligned'
        np.add.at(M, (y, P.argmax(1)), 1)
    return M
def rate(M, a, b): return 100 * M[SI[a], SI[b]] / M[SI[a]].sum()
PAIRS = [('Latin sink', [(s, 'Latin') for s in SC if s != 'Latin']),
         ('Headline scripts', [('Bengali-Assamese','Devanagari'), ('Gurmukhi','Devanagari'), ('Devanagari','Bengali-Assamese'), ('Devanagari','Gurmukhi')]),
         ('Telugu/Kannada', [('Telugu','Kannada'), ('Kannada','Telugu')])]
for tag, name in (('flatvit', 'flat ViT-B/16'), ('flat_effb0', 'EfficientNet-B0 word agent')):
    for sp in ('val', 'test'):
        M = conf(tag, sp)
        if M.sum() == 0: print('no files for', tag, sp); continue
        print(f'===== {name} | {sp} | words {int(M.sum()/5)} per seed | accuracy {100*np.trace(M)/M.sum():.2f}')
        for g, prs in PAIRS:
            print(g + ':', '; '.join(f'{a}->{b} {rate(M,a,b):.1f}%' for a, b in prs))
        E = M.copy(); np.fill_diagonal(E, 0)
        print('share of all errors going to Latin: %.1f%%' % (100*E[:, SI['Latin']].sum()/E.sum()))
        top = sorted([(E[i,j]/M[i].sum()*100, SC[i], SC[j]) for i in range(C) for j in range(C) if i != j], reverse=True)[:8]
        print('top 8 confusions (% of true class):', '; '.join(f'{a}->{b} {r:.1f}' for r, a, b in top))